# INPUT / LINEAGE — READ BEFORE RUN

## TYPE
Pretrained AutoShot inference on the **existing frozen AIC 2026 system**.
This refactors the legacy `autoshot-fn.ipynb`; it does not rebuild Stage03/20/Final24.

## INPUT A — Evidence 00 v0.2.1 executed output
Source: `00_FROZEN_SYSTEM_SNAPSHOT_QUERY_FREEZE_v0_2_1.ipynb`

Required authority:
- `freeze_manifest.json`
- `corpus_snapshot_manifest.parquet`
- 873 videos
- 177,321 official FrameRefs.

## INPUT B — raw Batch1
`nguyenminhtric/data-batch-1-aic`

## INPUT C — AutoShot weights
`anhnguynnhtinh/model-weights-autoshot`

Preferred official baseline filename:
`ckpt_0_200_0.pth`

The dataset name/file name is never trusted by itself. Before corpus inference the
notebook computes SHA-256 and verifies state-dict key/shape coverage against the exact
embedded `TransNetV2Supernet`.

## OUTPUT
- `per_video/*.json`
- `boundary_npz/*.npz`
- `shot_interval_registry.jsonl`
- `run_report.json`
- `component01_handoff.json`

## NEXT
`01 v2.2.3 PASS -> 02 BEiT3 shadow` or
`01 v2.2.3 PASS -> 01T AIC2026 self-training/domain-adaptation experiment`.

# AIC 2026 — Component 01: AutoShot PTS-aware Streaming — v2.2.3

**Thay thế notebook di sản `autoshot-fn(1).ipynb`.**

Sửa các lỗi P0:
- không decode toàn video vào RAM;
- mọi boundary/scene có `actual PTS`;
- không coi frame ordinal là thời gian;
- checkpoint load có coverage/shape gate, không silent partial load;
- neural workload bắt buộc dùng **cả cuda:0 và cuda:1**;
- output là *shot interval proposal*, không tự nhận là canonical `ShotRef`;
- lưu raw boundary probability để calibration về sau.

Output:
- `shot_interval_registry.jsonl`
- `per_video/<video_id>.json`
- `boundary_npz/<video_id>.npz`
- `run_report.json`

Canonical truth vẫn là `video_id + pts_time + raw video`.

## v2.2.3 — executed failure hotfix

Executed v2.2.2 reached the raw Batch1 preflight, then stopped before model loading with:

`EVIDENCE00_V0_2_1_OUTPUT_REQUIRED`

This is an **input-resolution failure**, not an AutoShot/model failure.

v2.2.3 changes:

1. performs a zero-install, zero-GPU preflight **before pip/model work**;
2. checks all three required parents at once:
   - raw Batch1,
   - AutoShot weights dataset,
   - executed Evidence-00 v0.2.1 output;
3. searches Evidence-00 only in non-raw/non-model Kaggle mounts, avoiding a recursive
   walk through the 350k+ raw corpus files;
4. accepts unpacked 00 output or a ZIP containing the exact
   `aic2026-evidence-frozen-system-snapshot-v0.2.1` freeze manifest;
5. when 00 is missing, fails with explicit Kaggle Add Input instructions rather than
   failing after dependency installation.

No AutoShot architecture, weights semantics, PTS policy, threshold, or GPU contract changes.

## v2.2.3 — model-weights dataset patch

- Pins `anhnguynnhtinh/model-weights-autoshot`.
- Prefers `ckpt_0_200_0.pth`, the baseline filename published by the AutoShot authors.
- Searches `.pth` only inside that dataset.
- Keeps explicit `AIC2026_AUTOSHOT_CKPT` override.
- Records checkpoint filename/size/SHA-256.
- Still fails closed on incompatible state-dict coverage.

## v2.2.3 runtime fix history

Executed Evidence-00 v0.2.1 is now the corpus authority.

Changes from 01 v2.2.0:

1. **P0 fixed:** `get_batches()` was referenced but undefined.
2. **P1 fixed:** removed broad raw `rglob("*")`; exactly 873 video paths now come
   from Evidence-00 `corpus_snapshot_manifest.parquet`.
3. **P1 lineage fixed:** AutoShot checkpoint is explicit/fail-closed; no fictional
   Kaggle checkpoint handle and no silent download.
4. **PTS validation upgraded:** full-video target PTS is sampled from the generated
   NPZ at 25/50/75%, rather than only the first 1200 decoded frames.
5. Resume now requires both per-video JSON and matching NPZ.
6. Strict two-GPU contract remains unchanged.

## Evidence-phase lock v2.2

Static PASS is not corpus/accuracy/latency PASS.
Seek semantics are locked: backward keyframe seek → decode forward → record landing PTS → choose first presentation frame at/after target → compare actual PTS to canonical target.

## v2.1 Pareto/runtime hardening

Không đổi AutoShot model hay threshold semantics.

Bổ sung benchmark:
- actual PTS seek round-trip;
- throughput video/hour;
- peak RAM/VRAM;
- latency p50/p95/p99;
- resume/idempotence.
- strict two-GPU contract vẫn giữ nguyên.

In [1]:
# ── PRE-FLIGHT: resolve all required Kaggle inputs before installs / GPU work ─
from pathlib import Path
import os, json, zipfile

PREFLIGHT_RAW_HANDLE="nguyenminhtric/data-batch-1-aic"
PREFLIGHT_MODEL_HANDLE="anhnguynnhtinh/model-weights-autoshot"
PREFLIGHT_E00_BUILD="aic2026-evidence-frozen-system-snapshot-v0.2.1"

def _dataset_root_candidates(owner,name):
    return [
        Path("/kaggle/input")/name,
        Path("/kaggle/input/datasets")/owner/name,
    ]

def _first_existing_dir(cands):
    hits=sorted({p.resolve() for p in cands if p.is_dir()},key=str)
    return hits[0] if hits else None

PREFLIGHT_RAW_ROOT=_first_existing_dir(
    _dataset_root_candidates("nguyenminhtric","data-batch-1-aic")
)
PREFLIGHT_MODEL_ROOT=_first_existing_dir(
    _dataset_root_candidates("anhnguynnhtinh","model-weights-autoshot")
)

def _valid_e00_manifest(path):
    try:
        d=json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None
    if (
        d.get("build_version")==PREFLIGHT_E00_BUILD
        and d.get("corpus_snapshot_freeze")=="PASS"
    ):
        return d
    return None

def _inspect_zip_for_e00(zp):
    try:
        with zipfile.ZipFile(zp,"r") as z:
            for info in z.infolist():
                if Path(info.filename).name!="freeze_manifest.json":
                    continue
                try:
                    d=json.loads(z.read(info).decode("utf-8"))
                except Exception:
                    continue
                if (
                    d.get("build_version")==PREFLIGHT_E00_BUILD
                    and d.get("corpus_snapshot_freeze")=="PASS"
                ):
                    return info.filename
    except zipfile.BadZipFile:
        return None
    return None

def _other_kaggle_mounts():
    root=Path("/kaggle/input")
    if not root.is_dir():
        return []
    roots=[]
    try:
        direct=[p.resolve() for p in root.iterdir() if p.is_dir()]
    except OSError:
        direct=[]
    excluded={x for x in (PREFLIGHT_RAW_ROOT,PREFLIGHT_MODEL_ROOT) if x is not None}
    for p in direct:
        # Direct Kaggle slug mount.
        if p not in excluded:
            roots.append(p)

    # New Kaggle /datasets/<owner>/<dataset> layout.
    droot=root/"datasets"
    if droot.is_dir():
        for owner_dir in droot.iterdir():
            if not owner_dir.is_dir():
                continue
            for ds in owner_dir.iterdir():
                if ds.is_dir() and ds.resolve() not in excluded:
                    roots.append(ds.resolve())
    return sorted(set(roots),key=str)

def _resolve_e00_preflight():
    explicit_root=os.environ.get("AIC2026_EVIDENCE00_ROOT","").strip()
    if explicit_root:
        root=Path(explicit_root).resolve()
        hits=[p for p in root.rglob("freeze_manifest.json") if p.is_file()]
        good=[p.resolve() for p in hits if _valid_e00_manifest(p) is not None]
        if len(good)!=1:
            raise RuntimeError({
                "message":"AIC2026_EVIDENCE00_ROOT_INVALID_OR_AMBIGUOUS",
                "root":str(root),
                "valid_manifests":[str(x) for x in good],
            })
        return {"kind":"manifest","path":good[0],"source":"EXPLICIT_ROOT"}

    explicit_zip=os.environ.get("AIC2026_EVIDENCE00_ZIP","").strip()
    if explicit_zip:
        zp=Path(explicit_zip).resolve()
        member=_inspect_zip_for_e00(zp) if zp.is_file() else None
        if member is None:
            raise RuntimeError({
                "message":"AIC2026_EVIDENCE00_ZIP_INVALID",
                "zip":str(zp),
                "required_build":PREFLIGHT_E00_BUILD,
            })
        return {"kind":"zip","path":zp,"member":member,"source":"EXPLICIT_ZIP"}

    manifest_hits=[]
    zip_hits=[]
    scanned_mounts=[]
    for root in _other_kaggle_mounts():
        scanned_mounts.append(str(root))
        # These mounts are expected to be small notebook-output/model-control datasets.
        # Raw/model roots were explicitly excluded above.
        for p in root.rglob("freeze_manifest.json"):
            if p.is_file() and _valid_e00_manifest(p) is not None:
                manifest_hits.append(p.resolve())
        for zp in root.rglob("*.zip"):
            if not zp.is_file():
                continue
            member=_inspect_zip_for_e00(zp)
            if member is not None:
                zip_hits.append((zp.resolve(),member))

    manifest_hits=sorted(set(manifest_hits),key=str)
    zip_hits=sorted(set(zip_hits),key=lambda x:(str(x[0]),x[1]))

    if len(manifest_hits)==1 and not zip_hits:
        return {"kind":"manifest","path":manifest_hits[0],"source":"OTHER_MOUNT_SCAN"}
    if not manifest_hits and len(zip_hits)==1:
        return {"kind":"zip","path":zip_hits[0][0],"member":zip_hits[0][1],"source":"OTHER_MOUNT_ZIP_SCAN"}
    if manifest_hits or zip_hits:
        raise RuntimeError({
            "message":"EVIDENCE00_MULTIPLE_VALID_CANDIDATES",
            "manifests":[str(x) for x in manifest_hits],
            "zips":[{"zip":str(z),"member":m} for z,m in zip_hits],
            "action":"set AIC2026_EVIDENCE00_ROOT or AIC2026_EVIDENCE00_ZIP explicitly",
        })

    return {
        "kind":"missing",
        "scanned_mounts":scanned_mounts,
    }

PREFLIGHT_E00=_resolve_e00_preflight()

missing=[]
if PREFLIGHT_RAW_ROOT is None:
    missing.append({
        "input":"RAW_BATCH1",
        "handle":PREFLIGHT_RAW_HANDLE,
        "action":"Kaggle > Add Input > dataset nguyenminhtric/data-batch-1-aic",
    })
if PREFLIGHT_MODEL_ROOT is None:
    missing.append({
        "input":"AUTOSHOT_WEIGHTS",
        "handle":PREFLIGHT_MODEL_HANDLE,
        "action":"Kaggle > Add Input > dataset anhnguynnhtinh/model-weights-autoshot",
    })
if PREFLIGHT_E00["kind"]=="missing":
    missing.append({
        "input":"EXECUTED_EVIDENCE00_v0.2.1_OUTPUT",
        "required_build":PREFLIGHT_E00_BUILD,
        "action":(
            "Kaggle > Add Input > Notebook Output/Dataset containing the executed "
            "00 v0.2.1 output (freeze_manifest.json + corpus_snapshot_manifest.parquet), "
            "or upload the 00 results ZIP as a Kaggle dataset."
        ),
    })

print("[PREFLIGHT]",{
    "raw":None if PREFLIGHT_RAW_ROOT is None else str(PREFLIGHT_RAW_ROOT),
    "model_weights":None if PREFLIGHT_MODEL_ROOT is None else str(PREFLIGHT_MODEL_ROOT),
    "evidence00":PREFLIGHT_E00,
})

if missing:
    raise RuntimeError({
        "message":"AUTOSHOT01_REQUIRED_INPUTS_MISSING",
        "missing":missing,
        "important":"Do NOT rerun Evidence-00. Attach its already executed output as an input.",
    })

[PREFLIGHT] {'raw': '/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic', 'model_weights': '/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot', 'evidence00': {'kind': 'manifest', 'path': PosixPath('/kaggle/input/notebooks/nguyenminhtric/0rr-note/aic2026_evidence_phase/00_frozen_system_snapshot_v021/freeze_manifest.json'), 'source': 'OTHER_MOUNT_SCAN'}}


In [2]:
# ── CELL 01: config / dependencies / immutable parent contracts ───────────────
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import importlib.util, subprocess, sys, os, json, hashlib, math, traceback
import zipfile, shutil, time
import numpy as np
import torch

def ensure_pkg(import_name,pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable,"-m","pip","install","--quiet",pip_name or import_name]
        )

for import_name,pip_name in (
    ("av","av"),("einops","einops"),("psutil","psutil"),
    ("pandas","pandas"),("pyarrow","pyarrow")
):
    ensure_pkg(import_name,pip_name)

import av
import psutil
import pandas as pd

BUILD_VERSION="aic2026-autoshot-pts-streaming-v2.2.3"
STAGE_ID="component01_autoshot_pts_streaming"

RAW_DATASET_HANDLE="nguyenminhtric/data-batch-1-aic"
MODEL_DATASET_HANDLE="anhnguynnhtinh/model-weights-autoshot"
OFFICIAL_AUTOSHOT_BASELINE_FILENAME="ckpt_0_200_0.pth"
EXPECTED_EVIDENCE00_BUILD="aic2026-evidence-frozen-system-snapshot-v0.2.1"
EXPECTED_VIDEO_COUNT=873
EXPECTED_OFFICIAL_FRAME_COUNT=177321
EXPECTED_DATASET_SNAPSHOT_ID="sha256:f9480c86372388b93a8c801357974adda22899846bb65f923ceb6c415b23cca5"
EXPECTED_CORPUS_ID="aic2026_batch1"
EXPECTED_FRAME_REGISTRY_SHA256="2186aa8f53a98a09063ec03b155a4fda93f15e2effbd583dd6cc45923ca04695"

BASE=Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK_ROOT=BASE/"aic2026_vnext"/"component01_autoshot"
PER_VIDEO=WORK_ROOT/"per_video"
BOUNDARY_NPZ=WORK_ROOT/"boundary_npz"
MATERIALIZED=WORK_ROOT/"materialized_inputs"
for p in (WORK_ROOT,PER_VIDEO,BOUNDARY_NPZ,MATERIALIZED):
    p.mkdir(parents=True,exist_ok=True)

SHOT_THRESHOLD=float(os.environ.get("AIC2026_SHOT_THRESHOLD","0.5"))
STREAM_CORE_FRAMES=int(os.environ.get("AIC2026_AUTOSHOT_CORE_FRAMES","2000"))
FORCE_REBUILD=os.environ.get("AIC2026_FORCE_REBUILD","0")=="1"
SEEK_ROUNDTRIP_TOL_SEC=float(os.environ.get("AIC2026_SEEK_TOL_SEC","0.20"))
CONTEXT_FRAMES=25
VIDEO_EXTENSIONS=(".mp4",".mkv",".mov",".avi",".webm",".m4v")

def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def resolve_raw_dataset():
    explicit=os.environ.get("AIC2026_VIDEO_ROOT","").strip()
    if explicit:
        p=Path(explicit).resolve()
        if not p.is_dir():
            raise RuntimeError(("AIC2026_VIDEO_ROOT_NOT_DIRECTORY",str(p)))
        return p,"EXPLICIT_ENV"

    candidates=[
        Path("/kaggle/input/data-batch-1-aic"),
        Path("/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic"),
    ]
    hits=sorted({p.resolve() for p in candidates if p.is_dir()},key=str)
    if not hits:
        raise RuntimeError({
            "message":"RAW_AIC2026_INPUT_REQUIRED",
            "dataset_handle":RAW_DATASET_HANDLE,
            "action":"Kaggle Add Input: nguyenminhtric/data-batch-1-aic",
        })
    return hits[0],"EXACT_KAGGLE_HANDLE"

RAW_ROOT,RAW_ROOT_SOURCE=resolve_raw_dataset()
print("[PASS] raw parent",{"handle":RAW_DATASET_HANDLE,"root":str(RAW_ROOT),"source":RAW_ROOT_SOURCE})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 43.9 MB/s eta 0:00:00
[PASS] raw parent {'handle': 'nguyenminhtric/data-batch-1-aic', 'root': '/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic', 'source': 'EXACT_KAGGLE_HANDLE'}


In [3]:
# ── CELL 02: materialize/verify the preflight-selected Evidence-00 parent ──────
def safe_member(name):
    p=Path(name)
    if p.is_absolute() or ".." in p.parts:
        raise RuntimeError(("ZIP_PATH_ESCAPE",name))
    return p

def materialize_e00_zip(zp):
    dest=MATERIALIZED/"evidence00"
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zp,"r") as z:
        bad=z.testzip()
        if bad is not None:
            raise RuntimeError(("ZIP_CRC_FAILED",str(zp),bad))
        for info in z.infolist():
            rel=safe_member(info.filename)
            target=(dest/rel).resolve()
            if dest.resolve()!=target and dest.resolve() not in target.parents:
                raise RuntimeError(("ZIP_PATH_ESCAPE",info.filename))
            if info.is_dir():
                target.mkdir(parents=True,exist_ok=True)
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(info,"r") as fi,target.open("wb") as fo:
                shutil.copyfileobj(fi,fo)

    hits=[]
    for p in dest.rglob("freeze_manifest.json"):
        try:
            d=json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if (
            d.get("build_version")==EXPECTED_EVIDENCE00_BUILD
            and d.get("corpus_snapshot_freeze")=="PASS"
        ):
            hits.append(p.resolve())
    hits=sorted(set(hits),key=str)
    if len(hits)!=1:
        raise RuntimeError(("EVIDENCE00_ZIP_VALID_MANIFEST_COUNT",len(hits),[str(x) for x in hits]))
    return hits[0]

if PREFLIGHT_E00["kind"]=="manifest":
    E00_FREEZE_PATH=Path(PREFLIGHT_E00["path"]).resolve()
    E00_SOURCE=PREFLIGHT_E00["source"]
elif PREFLIGHT_E00["kind"]=="zip":
    E00_FREEZE_PATH=materialize_e00_zip(Path(PREFLIGHT_E00["path"]).resolve())
    E00_SOURCE=PREFLIGHT_E00["source"]
else:
    raise RuntimeError("PREFLIGHT_EVIDENCE00_STATE_INVALID")

E00=json.loads(E00_FREEZE_PATH.read_text(encoding="utf-8"))
E00_ROOT=E00_FREEZE_PATH.parent

if E00.get("build_version")!=EXPECTED_EVIDENCE00_BUILD:
    raise RuntimeError(("EVIDENCE00_BUILD_DRIFT",E00.get("build_version"),EXPECTED_EVIDENCE00_BUILD))
if E00.get("corpus_snapshot_freeze")!="PASS":
    raise RuntimeError(("EVIDENCE00_CORPUS_FREEZE_NOT_PASS",E00.get("corpus_snapshot_freeze")))

for key,expected in (
    ("video_count",EXPECTED_VIDEO_COUNT),
    ("official_frame_count",EXPECTED_OFFICIAL_FRAME_COUNT),
):
    if int(E00.get(key,-1))!=expected:
        raise RuntimeError(("EVIDENCE00_COUNT_DRIFT",key,E00.get(key),expected))

if E00.get("dataset_snapshot_id")!=EXPECTED_DATASET_SNAPSHOT_ID:
    raise RuntimeError(("EVIDENCE00_SNAPSHOT_DRIFT",E00.get("dataset_snapshot_id")))
if E00.get("corpus_id")!=EXPECTED_CORPUS_ID:
    raise RuntimeError(("EVIDENCE00_CORPUS_DRIFT",E00.get("corpus_id")))
if E00.get("frame_registry_sha256")!=EXPECTED_FRAME_REGISTRY_SHA256:
    raise RuntimeError(("EVIDENCE00_FRAME_REGISTRY_DRIFT",E00.get("frame_registry_sha256")))

def rebase_e00_artifact(record,basename):
    expected_sha=str(record["sha256"])
    hits=sorted({p.resolve() for p in E00_ROOT.rglob(basename) if p.is_file()},key=str)
    if len(hits)!=1:
        raise RuntimeError(("EVIDENCE00_ARTIFACT_MISSING_OR_AMBIGUOUS",basename,[str(x) for x in hits]))
    p=hits[0]
    actual=sha256_file(p)
    if actual!=expected_sha:
        raise RuntimeError(("EVIDENCE00_ARTIFACT_SHA_MISMATCH",basename,actual,expected_sha))
    return p

SNAPSHOT_PATH=rebase_e00_artifact(
    E00["artifacts"]["corpus_snapshot_manifest"],
    "corpus_snapshot_manifest.parquet",
)
SNAPSHOT=pd.read_parquet(SNAPSHOT_PATH)

required_snapshot={"video_id","raw_relpath","dataset_snapshot_id","corpus_id","frame_registry_sha256"}
missing_cols=sorted(required_snapshot-set(SNAPSHOT.columns))
if missing_cols:
    raise RuntimeError(("EVIDENCE00_SNAPSHOT_SCHEMA_MISSING",missing_cols))
if len(SNAPSHOT)!=EXPECTED_VIDEO_COUNT or SNAPSHOT.video_id.astype(str).duplicated().any():
    raise RuntimeError(("EVIDENCE00_SNAPSHOT_VIDEO_ROWS",len(SNAPSHOT),
                        int(SNAPSHOT.video_id.astype(str).duplicated().sum())))

print("[PASS] Evidence00 parent",{
    "source":E00_SOURCE,
    "freeze_manifest":str(E00_FREEZE_PATH),
    "freeze_sha256":sha256_file(E00_FREEZE_PATH),
    "snapshot_sha256":sha256_file(SNAPSHOT_PATH),
    "videos":len(SNAPSHOT),
    "query_freeze":E00.get("query_manifest_freeze"),
})

[PASS] Evidence00 parent {'source': 'OTHER_MOUNT_SCAN', 'freeze_manifest': '/kaggle/input/notebooks/nguyenminhtric/0rr-note/aic2026_evidence_phase/00_frozen_system_snapshot_v021/freeze_manifest.json', 'freeze_sha256': 'c5aa00078be9b45c9a4d8c7161fb46d69245215be6466c30497307ca62be37c2', 'snapshot_sha256': 'd18c6e5731bf610e55546efe7ffb68565a8e19000633baee1608a516933048e0', 'videos': 873, 'query_freeze': 'PENDING_REAL_QUERY_WORKLOAD'}


In [4]:
# ── CELL 03: exact AutoShot weights dataset + frozen video list ───────────────
def resolve_model_dataset():
    explicit=os.environ.get("AIC2026_AUTOSHOT_MODEL_ROOT","").strip()
    if explicit:
        p=Path(explicit).resolve()
        if not p.is_dir():
            raise RuntimeError(("AIC2026_AUTOSHOT_MODEL_ROOT_NOT_DIRECTORY",str(p)))
        return p,"EXPLICIT_MODEL_ROOT"

    candidates=[
        Path("/kaggle/input/model-weights-autoshot"),
        Path("/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot"),
    ]
    hits=sorted({p.resolve() for p in candidates if p.is_dir()},key=str)
    if not hits:
        raise RuntimeError({
            "message":"AUTOSHOT_MODEL_DATASET_REQUIRED",
            "dataset_handle":MODEL_DATASET_HANDLE,
            "action":"Kaggle Add Input: anhnguynnhtinh/model-weights-autoshot",
        })
    hits=sorted(hits,key=lambda p:(0 if "/datasets/" in str(p) else 1,len(str(p))))
    return hits[0],"EXACT_KAGGLE_MODEL_DATASET"

def resolve_checkpoint():
    explicit=os.environ.get("AIC2026_AUTOSHOT_CKPT","").strip()
    if explicit:
        p=Path(explicit).resolve()
        if not p.is_file():
            raise RuntimeError(("AIC2026_AUTOSHOT_CKPT_MISSING",str(p)))
        return p,"EXPLICIT_CHECKPOINT",None

    model_root,root_source=resolve_model_dataset()
    pths=sorted({p.resolve() for p in model_root.rglob("*.pth") if p.is_file()},key=str)
    if not pths:
        raise RuntimeError(("AUTOSHOT_MODEL_DATASET_HAS_NO_PTH",str(model_root)))

    preferred=[p for p in pths if p.name==OFFICIAL_AUTOSHOT_BASELINE_FILENAME]
    if len(preferred)==1:
        return preferred[0],"OFFICIAL_BASELINE_FILENAME",{
            "model_dataset_root":str(model_root),
            "root_source":root_source,
            "pth_candidates":[str(x) for x in pths],
        }
    if len(preferred)>1:
        raise RuntimeError(("AUTOSHOT_OFFICIAL_FILENAME_DUPLICATE",[str(x) for x in preferred]))
    if len(pths)==1:
        return pths[0],"SINGLE_PTH_DATASET_FALLBACK",{
            "model_dataset_root":str(model_root),
            "root_source":root_source,
            "pth_candidates":[str(x) for x in pths],
        }
    raise RuntimeError({
        "message":"AUTOSHOT_CHECKPOINT_AMBIGUOUS",
        "preferred_filename":OFFICIAL_AUTOSHOT_BASELINE_FILENAME,
        "candidates":[str(x) for x in pths],
        "action":"set AIC2026_AUTOSHOT_CKPT",
    })

PRETRAINED_PATH,CHECKPOINT_SOURCE,MODEL_DATASET_DISCOVERY=resolve_checkpoint()
CHECKPOINT_SHA256=sha256_file(PRETRAINED_PATH)
CHECKPOINT_SIZE_BYTES=int(PRETRAINED_PATH.stat().st_size)

VIDEO_RECORDS=[]
for r in SNAPSHOT.sort_values("video_id").itertuples(index=False):
    p=(RAW_ROOT/str(r.raw_relpath)).resolve()
    if RAW_ROOT.resolve()!=p and RAW_ROOT.resolve() not in p.parents:
        raise RuntimeError(("RAW_PATH_ESCAPE",str(r.raw_relpath)))
    if not p.is_file():
        raise FileNotFoundError(("FROZEN_RAW_VIDEO_MISSING",str(r.video_id),str(p)))
    if p.stem!=str(r.video_id):
        raise RuntimeError(("VIDEO_ID_PATH_MISMATCH",str(r.video_id),p.stem,str(p)))
    VIDEO_RECORDS.append({"video_id":str(r.video_id),"raw_relpath":str(r.raw_relpath),"video_path":p})

if len(VIDEO_RECORDS)!=EXPECTED_VIDEO_COUNT:
    raise RuntimeError(("FROZEN_VIDEO_COUNT_DRIFT",len(VIDEO_RECORDS),EXPECTED_VIDEO_COUNT))

print("[PASS] AutoShot immutable inputs",{
    "videos":len(VIDEO_RECORDS),
    "model_dataset_handle":MODEL_DATASET_HANDLE,
    "checkpoint":str(PRETRAINED_PATH),
    "checkpoint_filename":PRETRAINED_PATH.name,
    "checkpoint_source":CHECKPOINT_SOURCE,
    "checkpoint_size_bytes":CHECKPOINT_SIZE_BYTES,
    "checkpoint_sha256":CHECKPOINT_SHA256,
})

[PASS] AutoShot immutable inputs {'videos': 873, 'model_dataset_handle': 'anhnguynnhtinh/model-weights-autoshot', 'checkpoint': '/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot/ckpt_0_200_0.pth', 'checkpoint_filename': 'ckpt_0_200_0.pth', 'checkpoint_source': 'OFFICIAL_BASELINE_FILENAME', 'checkpoint_size_bytes': 57243097, 'checkpoint_sha256': '3e85290546ce6d32f4a3581ec2cae87aedd2402246a0d46b4d361a330b4b1fa6'}


In [5]:
# ── CELL 04: AutoShot temporal batching contract + regression tests ────────────
AUTOSHOT_BATCH_FRAMES=100
AUTOSHOT_CONTEXT_FRAMES=25
AUTOSHOT_OUTPUT_STRIDE=50

def get_batches(frames):
    """
    TransNetV2-style context batching:
      25 repeated left context
      + original frames
      + 25 repeated right context
    then 100-frame windows at stride 50. The model's central [25:75] predictions
    reconstruct the original timeline exactly after truncation.
    """
    arr=np.asarray(frames)
    if arr.ndim!=4:
        raise ValueError(("AUTOSHOT_FRAMES_EXPECT_T_H_W_C",arr.shape))
    n=len(arr)
    if n==0:
        return

    padded=np.concatenate([
        np.repeat(arr[:1],AUTOSHOT_CONTEXT_FRAMES,axis=0),
        arr,
        np.repeat(arr[-1:],AUTOSHOT_CONTEXT_FRAMES,axis=0),
    ],axis=0)

    for start in range(0,n,AUTOSHOT_OUTPUT_STRIDE):
        batch=padded[start:start+AUTOSHOT_BATCH_FRAMES]
        if len(batch)<AUTOSHOT_BATCH_FRAMES:
            batch=np.concatenate([
                batch,
                np.repeat(batch[-1:],AUTOSHOT_BATCH_FRAMES-len(batch),axis=0),
            ],axis=0)
        if len(batch)!=AUTOSHOT_BATCH_FRAMES:
            raise AssertionError(("AUTOSHOT_BATCH_LENGTH",len(batch)))
        yield batch

def _batch_contract_regression():
    for n in (1,2,25,49,50,51,99,100,101,2001):
        # Encode original temporal ordinal in pixels.
        x=np.arange(n,dtype=np.int32)[:,None,None,None]
        x=np.repeat(x,3,axis=3)
        batches=list(get_batches(x))
        if len(batches)!=math.ceil(n/AUTOSHOT_OUTPUT_STRIDE):
            raise AssertionError(("BATCH_COUNT",n,len(batches)))
        if any(b.shape[0]!=AUTOSHOT_BATCH_FRAMES for b in batches):
            raise AssertionError(("BATCH_SHAPE",n,[b.shape for b in batches]))
        reconstructed=np.concatenate(
            [b[AUTOSHOT_CONTEXT_FRAMES:AUTOSHOT_CONTEXT_FRAMES+AUTOSHOT_OUTPUT_STRIDE]
             for b in batches],
            axis=0,
        )[:n]
        if not np.array_equal(reconstructed,x):
            raise AssertionError(("CENTRAL_TIMELINE_RECONSTRUCTION",n))
    return "PASS"

print("[PASS] AutoShot get_batches regression",_batch_contract_regression())

[PASS] AutoShot get_batches regression PASS


In [6]:
import torch.nn as nn
class Linear_(nn.Module):
    def __init__(self, in_features, out_features, bias=True, act="ReLU", is_folded=True):
        super(Linear_, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias
        self.act_type = act
        self.is_folded = is_folded
        self.linear = nn.Linear(in_features=self.in_features,
                                out_features=self.out_features,
                                bias=self.bias)
        self.act = _act(self.act_type)

    def forward(self, inputs):
        result_linear = self.linear(inputs)
        result = self.act(result_linear)
        return result

    @property
    def multiply_adds(self):
        result = self.in_features * self.out_features
        return result

    @property
    def params(self):
        params = self.in_features * self.out_features
        # TODO 不考虑fold方式，需要计算bias的参数量
        if self.bias is True and self.is_folded is False:
            params += self.out_features
            # print("%d" % self.out_features)
        return params


class Identity_(nn.Module):
    """
    skip connect
    """

    def __init__(self):
        super(Identity_, self).__init__()

    def forward(self, inputs):
        return inputs

def _act(act_type, **kwargs):
    if act_type is None or act_type == "Identity":
        return Identity_()
    elif act_type == "ReLU":
        result = nn.ReLU(inplace=True)
        return result
    else:
        raise Exception("Not implemented !!!")
    

    raise Exception("_act: Go here ..., maybe don\'t define some act")


In [7]:
import math

import numpy as np
import torch
import torch.nn as nn
from torch.nn import init
import torch.nn.functional as F
from einops import rearrange
from einops.layers.torch import Rearrange
from torch.nn.parallel import DataParallel


class TransNetV2Supernet(nn.Module):
    def __init__(self, D=1024,
                 use_many_hot_targets=True,
                 use_frame_similarity=True,
                 use_mean_pooling=None,
                 dropout_rate=0.5,
                 use_resnet_like_top=False,
                 frame_similarity_on_last_layer=False,
                 use_color_histograms=True,
                 chromosome=None):
        super(TransNetV2Supernet, self).__init__()

        self.reprocess_layer = (lambda x: x / 255.)
 
        # TODO Layer 1 - 6
        self.Layer_0_3 = DilatedDCNNV2(3, 16, multiplier=1)
        self.Layer_1_8 = DilatedDCNNV2ABC(16 * 4, 16, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_2_8 = DilatedDCNNV2ABC(16 * 4, 32, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_3_8 = DilatedDCNNV2ABC(32 * 4, 32, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_4_13 = DilatedDCNNV2(32 * 4, 64, multiplier=3, n_dilation=5)
        self.Layer_5_12 = DilatedDCNNV2(64 * 4, 64, multiplier=2, n_dilation=5)
        self.Layer_6_0 = Attention1D(dim_in=256 * 3 * 6, dim_out=1024, num_heads=4,
            qkv_bias=False,
            attn_drop=0.,
            proj_drop=0.,
            with_cls_token=False,
            skip_conv_proj=True,
            n_layer=0
        )

        self.pool = torch.nn.AvgPool3d(kernel_size=(1, 2, 2))

        if use_frame_similarity is True and use_color_histograms is True:
            in_features = 4864
        elif use_frame_similarity is True and use_color_histograms is False:
            in_features = 4736
        elif use_frame_similarity is False and use_color_histograms is True:
            in_features = 4736
        else:
            in_features = 4608

        self.fc1_0 = Linear_(in_features=in_features, out_features=D, bias=True, act="ReLU")
        in_features += 1024
        self.fc1 = Linear_(in_features=in_features, out_features=D, bias=True, act="ReLU")
        self.cls_layer1 = Linear_(in_features=1024, out_features=1, bias=True, act="Identity")
        self.cls_layer2 = Linear_(in_features=1024, out_features=1, bias=True, act="Identity") \
            if use_many_hot_targets else None
        self.frame_sim_layer = FrameSimilarity(in_channels=448, inner_channels=101) if use_frame_similarity else None
        self.color_hist_layer = ColorHistograms(lookup_window=101, output_dim=128) if use_color_histograms else None
        self.use_mean_pooling = use_mean_pooling
        self.dropout = torch.nn.Dropout(p=1.0 - dropout_rate) if dropout_rate is not None else None
        self.frame_similarity_on_last_layer = frame_similarity_on_last_layer
        self.resnet_like_top = use_resnet_like_top
        if self.resnet_like_top:
            raise Exception("Position resnet_like_top 1: should not be here !!!")
        # self.logits_act = torch.nn.Sigmoid()  # TODO put final activation here

        self._init_weights()
        
        

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                fan_out = m.weight.size(0)
                fan_in = m.weight.size(1)
                limit = math.sqrt(6.0 / (fan_in + fan_out))
                m.weight.data.uniform_(-limit, limit)
                if m.bias is not None:
                    m.bias.data.zero_()
            
            elif isinstance(m, nn.Conv3d):
                init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    def forward(self, inputs, **kwargs):
        x = self.reprocess_layer(inputs)

        if self.resnet_like_top:
            raise Exception("Position resnet_like_top 2: should not be here !!!")

        block_features = []
        shortcut = None
        # TODO Layer 1 ~ 6
        num_layers = 6
        for layer_index in range(num_layers):
            if layer_index == 0:
                op = self.Layer_0_3
            elif layer_index == 1:
                op = self.Layer_1_8
            elif layer_index == 2:
                op = self.Layer_2_8
            elif layer_index == 3:
                op = self.Layer_3_8
            elif layer_index == 4:
                op = self.Layer_4_13
            elif layer_index == 5:
                op = self.Layer_5_12

            x = op(x)
            if layer_index in [0, 2, 4]:
                shortcut = x
            else:
                x = shortcut + x
                x = self.pool(x)
                block_features.append(x)
        transf_x = self.Layer_6_0(x, t=x.size(1), h=3, w=6)

        if self.use_mean_pooling:
            x = torch.mean(x, dim=[3, 4])  # out is [BS, C, N]
        else:
            # TODO actually go here !
            x = x.permute(0, 2, 3, 4, 1)
            shape = [x.shape[0], x.shape[1], np.prod(x.shape[2:])]
            x = x.reshape(shape=shape)  # out is [BS, C, N * H * W]

        if self.frame_sim_layer is not None and not self.frame_similarity_on_last_layer:
            x = torch.cat([self.frame_sim_layer(block_features), x], dim=2)

        if self.color_hist_layer is not None:
            x = torch.cat([self.color_hist_layer(inputs), x], dim=2)
        
        if transf_x is not None:
            x = torch.cat([transf_x, x], dim=2)

            x = self.fc1(x)
        else:
            x = self.fc1_0(x)

        if self.dropout is not None:
            x = self.dropout(x)

        if self.frame_sim_layer is not None and self.frame_similarity_on_last_layer:
            x = torch.cat([self.frame_sim_layer(block_features), x], dim=2)

        one_hot = self.cls_layer1(x)
        # one_hot = self.logits_act(one_hot)

        many_hot = None
        if self.cls_layer2 is not None:
            many_hot = self.cls_layer2(x)
            # many_hot = self.logits_act(self.cls_layer2(x))

        if many_hot != None:
            return one_hot, many_hot
        return one_hot


def gather_nd(params, indices):
    """ The same as tf.gather_nd but batched gather is not supported yet.
    indices is an k-dimensional integer tensor, best thought of as a (k-1)-dimensional tensor of indices into params, where each element defines a slice of params:

    output[\\(i_0, ..., i_{k-2}\\)] = params[indices[\\(i_0, ..., i_{k-2}\\)]]

    Args:
        params (Tensor): "n" dimensions. shape: [x_0, x_1, x_2, ..., x_{n-1}]
        indices (Tensor): "k" dimensions. shape: [y_0,y_2,...,y_{k-2}, m]. m <= n.

    Returns: gathered Tensor.
        shape [y_0,y_2,...y_{k-2}] + params.shape[m:]

    """
    orig_shape = list(indices.shape)
    num_samples = np.prod(orig_shape[:-1])
    m = orig_shape[-1]
    n = len(params.shape)

    if m <= n:
        out_shape = orig_shape[:-1] + list(params.shape)[m:]
    else:
        raise ValueError(f'the last dimension of indices must less or equal to the rank of params. '
                         f'Got indices:{indices.shape}, params:{params.shape}. {m} > {n}')

    indices = indices.reshape((num_samples, m)).transpose(0, 1).tolist()
    output = params[indices]  # (num_samples, ...)
    return output.reshape(out_shape).contiguous()


class FrameSimilarity(nn.Module):

    def __init__(self,
                 in_channels,
                 inner_channels,
                 similarity_dim=128,
                 lookup_window=101,
                 output_dim=128,
                 stop_gradient=False,
                 use_bias=True):
        super(FrameSimilarity, self).__init__()

        self.projection = Linear_(in_features=in_channels, out_features=similarity_dim,
                                  bias=use_bias, act="Identity")
        self.fc = Linear_(in_features=inner_channels, out_features=output_dim, bias=True, act="ReLU")

        self.lookup_window = lookup_window
        self.stop_gradient = stop_gradient
        assert lookup_window % 2 == 1, "`lookup_window` must be odd integer"
        if torch.cuda.is_available() is True:
            self.device = "cuda"
        else:
            self.device = "cpu"

    def forward(self, inputs):
        # pt version [BS, C, N, H, W], so [3, 4] means apply avg on spatial dim, out dim is [BS, C, N]
        x = torch.cat([torch.mean(x, dim=[3, 4]) for x in inputs], dim=1)

        if self.stop_gradient:
            x = x.detach()

        x = x.permute(dims=[0, 2, 1])  # out is [BS, N ,C]
        batch_size, time_window, old_channels = x.shape
        x = x.reshape(shape=[batch_size * time_window, old_channels])  # [BS X N, C]
        x = self.projection(x)
        x = F.normalize(x, p=2, dim=1)  # norm at C dim

        _, new_channels = x.shape
        x = x.reshape(shape=[batch_size, time_window, new_channels])
        y = x.permute(dims=[0, 2, 1])
        similarities = torch.matmul(x, y)  # [batch_size, time_window, time_window]
        # note that it operates on dimensions of the input tensor in a backward fashion (from last dimension to the first dimension)
        similarities_padded = F.pad(similarities,
                                    pad=[(self.lookup_window - 1) // 2, (self.lookup_window - 1) // 2, 0, 0, 0, 0])

        batch_indices = torch.arange(0, batch_size, device=self.device). \
            reshape(shape=[batch_size, 1, 1]). \
            repeat([1, time_window, self.lookup_window])
        time_indices = torch.arange(0, time_window, device=self.device). \
            reshape(shape=[1, time_window, 1]). \
            repeat([batch_size, 1, self.lookup_window])
        lookup_indices = torch.arange(0, self.lookup_window, device=self.device). \
                             reshape(shape=[1, 1, self.lookup_window]). \
                             repeat([batch_size, time_window, 1]) + time_indices

        indices = torch.stack([batch_indices, time_indices, lookup_indices], dim=-1)

        similarities = gather_nd(similarities_padded, indices)
        return self.fc(similarities)

class ColorHistograms(nn.Module):
    
    def __init__(self, lookup_window=101, output_dim=128):
        super(ColorHistograms, self).__init__()
        self.fc = Linear_(in_features=101, out_features=output_dim, bias=True, act="ReLU") \
            if output_dim is not None else None
        self.lookup_window = lookup_window
        assert lookup_window % 2 == 1, "`lookup_window` must be odd integer"
        if torch.cuda.is_available() is True:
            self.device = "cuda"
        else:
            self.device = "cpu"

    def unsorted_segment_sum(self, data, segment_ids, num_segments):
        """
        https://gist.github.com/bbrighttaer/207dc03b178bbd0fef8d1c0c1390d4be
        """
        assert all([i in data.shape for i in segment_ids.shape]), "segment_ids.shape should be a prefix of data.shape"
        # segment_ids is a 1-D tensor repeat it to have the same shape as data
        if len(segment_ids.shape) == 1:
            s = torch.prod(torch.tensor(data.shape[1:])).long()
            segment_ids = segment_ids.repeat_interleave(s).view(segment_ids.shape[0], *data.shape[1:])
        assert data.shape == segment_ids.shape, "data.shape and segment_ids.shape should be equal"
        shape = [num_segments] + list(data.shape[1:])
        tensor = torch.zeros(*shape, device=self.device).scatter_add(0, segment_ids, data.float())
        tensor = tensor.type(data.dtype)
        return tensor

    def compute_color_histograms(self, frames):
        frames = frames.type(dtype=torch.int32)
        # pt version [BS, C, N, H, W]  ---> tf version [BS, N, H, W, C]
        frames = frames.permute(0, 2, 3, 4, 1)

        def get_bin(frames):
            # returns 0 .. 511
            R, G, B = frames[:, :, 0], frames[:, :, 1], frames[:, :, 2]
            R, G, B = R >> 5, G >> 5, B >> 5
            return (R << 6) + (G << 3) + B

        batch_size, time_window, height, width = frames.shape[0], frames.shape[1], frames.shape[2], frames.shape[3]

        no_channels = frames.shape[-1]
        assert no_channels == 3 or no_channels == 6
        if no_channels == 3:
            frames_flatten = frames.reshape(shape=[batch_size * time_window, height * width, 3])
        else:
            frames_flatten = frames.reshape(shape=[batch_size * time_window, height * width * 2, 3])

        binned_values = get_bin(frames_flatten)
        frame_bin_prefix = torch.arange(0, batch_size * time_window, dtype=torch.int32, device=self.device) << 9
        frame_bin_prefix = frame_bin_prefix.unsqueeze(dim=-1)
        binned_values = binned_values + frame_bin_prefix

        ones = torch.ones_like(binned_values, dtype=torch.int32, device=self.device)
        histograms = self.unsorted_segment_sum(data=ones,
                                               segment_ids=binned_values.type(dtype=torch.long),
                                               num_segments=batch_size * time_window * 512)
        histograms = torch.sum(histograms, dim=1)
        histograms = histograms.reshape(shape=[batch_size, time_window, 512])

        histograms_normalized = histograms.type(dtype=torch.float32)
        histograms_normalized = histograms_normalized / torch.norm(histograms_normalized, dim=2, keepdim=True)
        return histograms_normalized

    def forward(self, inputs):
        x = self.compute_color_histograms(inputs)

        batch_size, time_window = x.shape[0], x.shape[1]
        y = x.permute(dims=[0, 2, 1])
        similarities = torch.matmul(x, y)  # [batch_size, time_window, time_window]
        # note that it operates on dimensions of the input tensor in a backward fashion (from last dimension to the first dimension)
        similarities_padded = F.pad(similarities,
                                    pad=[(self.lookup_window - 1) // 2, (self.lookup_window - 1) // 2, 0, 0, 0, 0])

        batch_indices = torch.arange(0, batch_size, device=self.device). \
            reshape(shape=[batch_size, 1, 1]). \
            repeat([1, time_window, self.lookup_window])
        time_indices = torch.arange(0, time_window, device=self.device). \
            reshape(shape=[1, time_window, 1]). \
            repeat([batch_size, 1, self.lookup_window])
        lookup_indices = torch.arange(0, self.lookup_window, device=self.device). \
                             reshape(shape=[1, 1, self.lookup_window]). \
                             repeat([batch_size, time_window, 1]) + time_indices

        indices = torch.stack([batch_indices, time_indices, lookup_indices], dim=-1)

        similarities = gather_nd(similarities_padded, indices)

        if self.fc is not None:
            return self.fc(similarities)
        return similarities


class ConvexCombinationRegularization(nn.Module):

    def __init__(self, ):
        super(ConvexCombinationRegularization, self).__init__()

        raise Exception("ConvexCombinationRegularization: should not init this class !!!")

    def forward(self, inputs):
        pass


class DilatedDCNNV2ABC(nn.Module):
    def __init__(self, in_channels,
                 filters,
                 batch_norm=True,
                 activation=F.relu,
                 octave_conv=False,
                 multiplier=4,
                 n_dilation=4,
                 st_type="A"):
        super(DilatedDCNNV2ABC, self).__init__()
        assert not (octave_conv and batch_norm)

        self.share = torch.nn.Conv3d(in_channels=in_channels,
                                     out_channels=multiplier * filters,
                                     kernel_size=(1, 3, 3),
                                     padding=(0, 1, 1),
                                     dilation=(1, 1, 1),
                                     bias=False)
        init.kaiming_normal_(self.share.weight, mode="fan_in", nonlinearity="relu")

        self.conv_blocks = nn.ModuleList()

        n_in_plane = multiplier * filters
        if st_type == "B":
            n_in_plane = in_channels

        n_filter_per_module = (filters * 4) // n_dilation  # multiplier
        for dilation in range(n_dilation-1):
            self.conv_blocks.append(
                Conv3DConfigurable(
                    n_in_plane,
                    n_filter_per_module,
                    2 ** dilation,
                    mid_filter=n_in_plane,
                    separable=True,
                    sharable=True,
                    use_bias=not batch_norm,
                    octave=octave_conv
                )
            )
        self.conv_blocks.append(
            Conv3DConfigurable(
                n_in_plane,
                (filters * 4) - n_filter_per_module * (n_dilation-1),  # multiplier
                2 ** (n_dilation - 1),
                mid_filter=n_in_plane,
                separable=True,
                sharable=True,
                use_bias=not batch_norm,
                octave=octave_conv
            )
        )

        self.octave = octave_conv
        self.multiplier = multiplier
        self.n_dilation = n_dilation
        self.st_type = st_type

        self.batch_norm = torch.nn.BatchNorm3d(
            num_features=filters * 4,  # multiplier,
            eps=1e-3, momentum=0.1) if batch_norm else None
        self.activation = activation

    def forward(self, inputs):
        feature = self.share(inputs)
        if self.st_type == "A":
            x = []
#             print(len(self.conv_blocks), feature.size())
            for block in self.conv_blocks:
#                 print(block)
                x.append(block(feature))
            x = torch.cat(x, dim=1)
        elif self.st_type == "B":
            x = []
            for block in self.conv_blocks:
                x.append(block(inputs))
            x = torch.cat(x, dim=1)
            x = x + feature
        elif self.st_type == "C":
            x = []
            for block in self.conv_blocks:
                x.append(block(feature))
            x = torch.cat(x, dim=1)
            x = x + feature
        else:
            raise Exception("Not Implemented ST Type" + self.st_type)

        if self.octave:
            raise Exception("Position octave 1: should not be here !!!")

        if self.batch_norm is not None:
            x = self.batch_norm(x)

        if self.activation is not None:
            if self.octave:
                raise Exception("Position octave 2: should not be here !!!")
            else:
                x = self.activation(x)
        return x


class DilatedDCNNV2(nn.Module):
    def __init__(self, in_channels,
                 filters,
                 multiplier=2,
                 n_dilation=4,
                 batch_norm=True,
                 activation=F.relu,
                 octave_conv=False):
        super(DilatedDCNNV2, self).__init__()
        assert not (octave_conv and batch_norm)

        self.n_dilation = n_dilation
        self.conv_blocks = nn.ModuleList()


        n_filter_per_module = (filters * 4) // n_dilation  # multiplier
        for dilation in range(n_dilation-1):
            self.conv_blocks.append(
                Conv3DConfigurable(
                    in_channels,
                    n_filter_per_module,
                    mid_filter=multiplier*filters,
                    dilation_rate=2 ** dilation,
                    use_bias=not batch_norm,
                    octave=octave_conv
                )
            )
        self.conv_blocks.append(
            Conv3DConfigurable(
                in_channels,
                (filters * 4) - n_filter_per_module * (n_dilation-1),  # multiplier
                mid_filter=multiplier*filters,
                dilation_rate=2 ** (n_dilation-1),
                use_bias=not batch_norm,
                octave=octave_conv
            )
        )

        self.batch_norm = torch.nn.BatchNorm3d(num_features=filters * 4, eps=1e-3, momentum=0.1) if batch_norm else None
        self.activation = activation
        self.octave = octave_conv

    def forward(self, inputs):
        x = []
        for block in self.conv_blocks:
            x.append(block(inputs))
        x = torch.cat(x, dim=1)

        if self.octave:
            raise Exception("Position octave 1: should not be here !!!")


        if self.batch_norm is not None:
            x = self.batch_norm(x)

        if self.activation is not None:
            if self.octave:
                raise Exception("Position octave 2: should not be here !!!")
            else:
                x = self.activation(x)
        return x


class Conv3DConfigurable(nn.Module):
    def __init__(self, in_channels,
                 filters,
                 dilation_rate,
                 mid_filter=None,
                 separable=True,
                 sharable=False,
                 octave=False,
                 use_bias=False):
        super(Conv3DConfigurable, self).__init__()
        assert not (separable and octave)

        if separable:
            # (2+1)D convolution https://arxiv.org/pdf/1711.11248.pdf
            from torch.nn import init
            self.layers = nn.ModuleList()
            if not sharable:
                conv1 = torch.nn.Conv3d(in_channels=in_channels,
                                        out_channels=2 * filters if mid_filter is None else mid_filter,
                                        kernel_size=(1, 3, 3),
                                        padding=(0, 1, 1),
                                        dilation=(1, 1, 1),
                                        bias=False)
                init.kaiming_normal_(conv1.weight, mode="fan_in", nonlinearity="relu")
                self.layers.append(conv1)

            conv2 = torch.nn.Conv3d(in_channels=2 * filters if mid_filter is None else mid_filter,
                                    out_channels=filters,
                                    kernel_size=(3, 1, 1),
                                    padding=(1 * dilation_rate, 0, 0),
                                    dilation=(dilation_rate, 1, 1),
                                    bias=use_bias)
            init.kaiming_normal_(conv2.weight, mode="fan_in", nonlinearity="relu")
            self.layers.append(conv2)
        elif octave:
            raise Exception("Positon octave 3: should not be here !!!")
        else:
            raise Exception("Positon else 1: should not be here !!!")

    def forward(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x
    
    
class Attention1D(nn.Module):
    def __init__(self,
                 dim_in,
                 dim_out,
                 num_heads,
                 qkv_bias=False,
                 attn_drop=0.,
                 proj_drop=0.,
                 with_cls_token=False,
                 skip_conv_proj=False,
                 n_layer=1,
                 **kwargs
                 ):
        super().__init__()
        self.dim = dim_out
        self.num_heads = num_heads
        self.n_layer = n_layer
        # head_dim = self.qkv_dim // num_heads
        self.scale = dim_out ** -0.5
        self.with_cls_token = with_cls_token
        
        self.proj_q = nn.ModuleList()
        self.proj_k = nn.ModuleList()
        self.proj_v = nn.ModuleList()
        self.attn_drop = nn.ModuleList()
        self.proj = nn.ModuleList()
        self.proj_drop = nn.ModuleList()
        
        for _ in range(n_layer):
            self.proj_q.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))
            self.proj_k.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))
            self.proj_v.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))

            self.attn_drop.append(nn.Dropout(attn_drop))
            self.proj.append(nn.Linear(dim_out, dim_out))
            self.proj_drop.append(nn.Dropout(proj_drop))
            
            dim_in = dim_out

    def forward(self, x, t, h, w):
        x = rearrange(x, 'b c t H W -> b t (c H W)')
        if self.n_layer == 0:
            return None
        
        for idx in range(self.n_layer):
            q = rearrange(self.proj_q[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)
            k = rearrange(self.proj_k[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)
            v = rearrange(self.proj_v[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)

            attn_score = torch.einsum('bhlk,bhtk->bhlt', [q, k]) * self.scale
            attn = F.softmax(attn_score, dim=-1)
            attn = self.attn_drop[idx](attn)

            x = torch.einsum('bhlt,bhtv->bhlv', [attn, v])
            x = rearrange(x, 'b h t d -> b t (h d)')

            x = self.proj[idx](x)
            x = self.proj_drop[idx](x)
        return x

In [8]:
# ── CELL 04: hardened AutoShot loader + predictor ──────────────────────────────
class AutoShotV2:
    def __init__(self, pretrained_path, device):
        self.device=str(device)
        self.model=self._load(pretrained_path)

    def _load(self, pretrained_path):
        p=Path(pretrained_path)
        if not p.exists():
            raise FileNotFoundError(p)
        model=TransNetV2Supernet().eval()
        raw=torch.load(p,map_location="cpu",weights_only=True)
        state=raw.get("net",raw)
        target=model.state_dict()

        usable={}
        shape_mismatch=[]
        for k,v in state.items():
            if k in target:
                if tuple(v.shape)==tuple(target[k].shape):
                    usable[k]=v
                else:
                    shape_mismatch.append((k,tuple(v.shape),tuple(target[k].shape)))

        coverage=len(usable)/max(1,len(target))
        missing=sorted(set(target)-set(usable))
        if shape_mismatch or coverage<0.995:
            raise RuntimeError({
                "message":"AUTOSHOT_CHECKPOINT_COVERAGE_FAILED",
                "coverage":coverage,
                "missing_preview":missing[:20],
                "shape_mismatch_preview":shape_mismatch[:10],
            })
        model.load_state_dict(usable,strict=False)
        model=model.to(self.device).eval()
        print("[PASS] AutoShot checkpoint",self.device,{"coverage":coverage,"keys":len(usable)})
        return model

    @torch.no_grad()
    def predict_batch(self,batch):
        x=torch.from_numpy(batch.transpose((3,0,1,2))[None,...]).float().to(self.device)
        y=self.model(x)
        if isinstance(y,tuple): y=y[0]
        out=torch.sigmoid(y[0]).float().reshape(-1).cpu().numpy()
        if len(out)!=len(batch):
            raise RuntimeError(("AUTOSHOT_MODEL_OUTPUT_LENGTH",len(out),len(batch)))
        return out

    def detect_shots(self,frames):
        preds=[]
        for batch in get_batches(frames):
            p=self.predict_batch(batch)
            preds.append(p[25:75])
        return np.concatenate(preds,axis=0)[:len(frames)]

In [9]:
# ── CELL 05: streaming PyAV decode with actual PTS ─────────────────────────────
def iter_video_lowres(video_path,width=48,height=27):
    container=av.open(str(video_path))
    try:
        stream=container.streams.video[0]
        tb_q=stream.time_base
        tb=float(tb_q)
        stream_start_time_raw=None if stream.start_time is None else int(stream.start_time)
        stream_start_time_sec=None if stream.start_time is None else float(stream.start_time)*tb
        ordinal=0
        last_pts_time=-math.inf
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            pts_time=float(frame.pts)*tb
            if pts_time+1e-9<last_pts_time:
                raise RuntimeError(("PTS_NON_MONOTONIC",str(video_path),last_pts_time,pts_time))
            last_pts_time=pts_time
            rgb=frame.reformat(width=width,height=height,format="rgb24").to_ndarray()
            yield rgb,{
                "decode_ordinal":ordinal,
                "pts_raw":int(frame.pts),
                "pts_time":pts_time,
                "time_base":tb,
                "time_base_num":int(tb_q.numerator),
                "time_base_den":int(tb_q.denominator),
                "stream_start_time_raw":stream_start_time_raw,
                "stream_start_time_sec":stream_start_time_sec,
                "presentation_time_normalized":pts_time,
                "missing_pts_policy":"SKIP_FRAME_AND_COUNT",
                "duplicate_pts_policy":"ALLOW_DUPLICATE_PTS_IF_DECODE_ORDINAL_INCREASES",
                "non_monotonic_pts_policy":"FAIL_CLOSED",
            }
            ordinal+=1
    finally:
        container.close()

def streaming_predictions(detector,video_path):
    left_frames=[]
    buf_frames=[]
    buf_meta=[]
    rows=[]

    def consume(core_frames,core_meta,right_frames):
        window=left_frames+core_frames+right_frames
        arr=np.stack(window,axis=0)
        pred=detector.detect_shots(arr)
        off=len(left_frames)
        core_pred=pred[off:off+len(core_frames)]
        return [
            {**m,"boundary_probability":float(p)}
            for m,p in zip(core_meta,core_pred)
        ]

    for frame,meta in iter_video_lowres(video_path):
        buf_frames.append(frame); buf_meta.append(meta)
        if len(buf_frames)>=STREAM_CORE_FRAMES+CONTEXT_FRAMES:
            core_frames=buf_frames[:STREAM_CORE_FRAMES]
            core_meta=buf_meta[:STREAM_CORE_FRAMES]
            right=buf_frames[STREAM_CORE_FRAMES:STREAM_CORE_FRAMES+CONTEXT_FRAMES]
            rows.extend(consume(core_frames,core_meta,right))
            left_frames=core_frames[-CONTEXT_FRAMES:]
            buf_frames=buf_frames[STREAM_CORE_FRAMES:]
            buf_meta=buf_meta[STREAM_CORE_FRAMES:]

    if buf_frames:
        rows.extend(consume(buf_frames,buf_meta,[]))
    return rows

def prediction_rows_to_scenes(rows,threshold=SHOT_THRESHOLD):
    if not rows: return []
    flags=np.array([r["boundary_probability"]>threshold for r in rows],dtype=np.uint8)
    spans=[]
    start=0
    prev=0
    for i,t in enumerate(flags):
        if prev==0 and t==1 and i!=0:
            spans.append((start,i))
        if prev==1 and t==0:
            start=i
        prev=int(t)
    if start<len(rows):
        spans.append((start,len(rows)-1))
    if not spans:
        spans=[(0,len(rows)-1)]

    scenes=[]
    for j,(a,b) in enumerate(spans):
        ra,rb=rows[a],rows[b]
        lo=max(0,b-2); hi=min(len(rows),b+3)
        scenes.append({
            "shot_proposal_id":f"shot_{j:06d}",
            "start_decode_ordinal":int(ra["decode_ordinal"]),
            "end_decode_ordinal":int(rb["decode_ordinal"]),
            "start_pts_raw":int(ra["pts_raw"]),
            "end_pts_raw":int(rb["pts_raw"]),
            "start_pts_time":float(ra["pts_time"]),
            "end_pts_time":float(rb["pts_time"]),
            "duration_sec":max(0.0,float(rb["pts_time"]-ra["pts_time"])),
            "boundary_probability_local_max":float(max(x["boundary_probability"] for x in rows[lo:hi])),
            "threshold_used":float(threshold),
            "canonical_status":"SHOT_PROPOSAL_NOT_CANONICAL",
        })
    return scenes

In [10]:
# ── CELL 06: per-video processing + provenance ─────────────────────────────────
def stable_video_id(video_path,root):
    rel=Path(video_path).resolve().relative_to(Path(root).resolve())
    return Path(rel).stem,rel.as_posix()


def process_video_v2(video_path,root,detector,gpu_id):
    video_id,rel=stable_video_id(video_path,root)
    cached=PER_VIDEO/f"{video_id}.json"
    cached_npz=BOUNDARY_NPZ/f"{video_id}.npz"

    if cached.is_file() and cached_npz.is_file() and not FORCE_REBUILD:
        try:
            prior=json.loads(cached.read_text(encoding="utf-8"))
            z=np.load(cached_npz)
            cached_pts=np.asarray(z["pts_time"],dtype=np.float64)
            cached_probs=np.asarray(z["boundary_probability"],dtype=np.float32)
            cache_valid=(
                prior.get("build_version")==BUILD_VERSION
                and prior.get("pts_monotonic") is True
                and len(cached_pts)==int(prior.get("frame_count",-1))
                and len(cached_probs)==len(cached_pts)
                and len(cached_pts)>0
                and np.isfinite(cached_pts).all()
                and np.all(np.diff(cached_pts)>=-1e-9)
            )
            if cache_valid:
                return {
                    **{k:v for k,v in prior.items() if k!="scenes"},
                    "resume_status":"REUSED_VALID_JSON_AND_NPZ",
                }
        except Exception:
            pass

    t0=time.perf_counter()
    rss0=psutil.Process().memory_info().rss
    torch.cuda.reset_peak_memory_stats(torch.device(gpu_id))
    rows=streaming_predictions(detector,video_path)
    if not rows:
        raise RuntimeError(("NO_DECODED_FRAMES",str(video_path)))

    scenes=prediction_rows_to_scenes(rows)
    pts=np.asarray([r["pts_time"] for r in rows],dtype=np.float64)
    probs=np.asarray([r["boundary_probability"] for r in rows],dtype=np.float32)
    ords=np.asarray([r["decode_ordinal"] for r in rows],dtype=np.int64)
    pts_raw=np.asarray([r["pts_raw"] for r in rows],dtype=np.int64)

    if not np.isfinite(pts).all() or not np.all(np.diff(pts)>=-1e-9):
        raise RuntimeError(("PTS_ARRAY_INVALID",video_id))

    np.savez_compressed(
        cached_npz,
        decode_ordinal=ords,
        pts_raw=pts_raw,
        pts_time=pts,
        boundary_probability=probs,
    )

    payload={
        "schema_version":"autoshot-pts-v2",
        "build_version":BUILD_VERSION,
        "video_id":video_id,
        "source_relpath":rel,
        "video_path":str(video_path),
        "gpu":gpu_id,
        "frame_count":len(rows),
        "pts_monotonic":True,
        "pts_time_min":float(pts.min()),
        "pts_time_max":float(pts.max()),
        "boundary_probability_mean":float(probs.mean()),
        "boundary_probability_p95":float(np.quantile(probs,0.95)),
        "scene_count":len(scenes),
        "scenes":scenes,
        "time_truth":"ACTUAL_DECODED_PTS",
        "mapping_reconciliation_status":"BOUND_TO_EVIDENCE00_FROZEN_VIDEO_UNIVERSE",
        "canonical_shot_status":"NOT_CREATED_PROPOSALS_ONLY",
        "checkpoint_sha256":CHECKPOINT_SHA256,
        "evidence00_freeze_sha256":sha256_file(E00_FREEZE_PATH),
        "elapsed_sec":float(time.perf_counter()-t0),
        "process_rss_delta_bytes_approx":int(psutil.Process().memory_info().rss-rss0),
        "cuda_peak_bytes":int(torch.cuda.max_memory_allocated(torch.device(gpu_id))),
        "resume_status":"BUILT",
    }
    cached.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
    return {k:v for k,v in payload.items() if k!="scenes"}

In [11]:
SEEK_PRESENTATION_POLICY='FIRST_PRESENTATION_AT_OR_AFTER_TARGET'
def seek_decode_select(video_path,target_sec,max_frames=200):
    container=av.open(str(video_path))
    try:
        stream=container.streams.video[0]; tb_q=stream.time_base; tb=float(tb_q)
        start_raw=None if stream.start_time is None else int(stream.start_time)
        start_sec=None if stream.start_time is None else float(stream.start_time)*tb
        container.seek(max(0,int(target_sec*1_000_000)),any_frame=False,backward=True)
        landing=None; selected=None; ordinal=0; missing=0; duplicate=0; prev=None
        for frame in container.decode(stream):
            if frame.pts is None: missing+=1; continue
            raw=int(frame.pts); t=float(frame.pts)*tb
            if landing is None: landing={'pts_raw':raw,'pts_time':t,'decode_ordinal_after_seek':ordinal}
            if prev is not None and raw==prev: duplicate+=1
            prev=raw
            if t+1e-12>=float(target_sec): selected={'pts_raw':raw,'pts_time':t,'decode_ordinal_after_seek':ordinal}; break
            ordinal+=1
            if ordinal>=max_frames: break
        return {'target_pts_time':float(target_sec),'time_base_num':int(tb_q.numerator),'time_base_den':int(tb_q.denominator),'time_base':tb,'stream_start_time_raw':start_raw,'stream_start_time_sec':start_sec,'seek_landing':landing,'selected':selected,'missing_pts_count':missing,'duplicate_pts_count':duplicate,'selection_policy':SEEK_PRESENTATION_POLICY}
    finally: container.close()


def pts_roundtrip_probe(video_path,target_pts):
    pts=np.asarray(target_pts,dtype=np.float64)
    pts=pts[np.isfinite(pts)]
    if len(pts)<3:
        return {"status":"N/A_TOO_SHORT","target_count":int(len(pts))}

    idx=[len(pts)//4,len(pts)//2,(3*len(pts))//4]
    targets=[float(pts[i]) for i in idx]
    rec=[]
    for target in targets:
        z=seek_decode_select(video_path,target)
        if z["selected"] is None:
            return {"status":"FAIL_NO_SELECTED_PRESENTATION_FRAME","record":z}
        signed=float(z["selected"]["pts_time"]-target)
        z["signed_drift_sec"]=signed
        z["absolute_drift_sec"]=abs(signed)
        rec.append(z)

    mx=max(z["absolute_drift_sec"] for z in rec)
    return {
        "status":"PASS" if mx<=SEEK_ROUNDTRIP_TOL_SEC else "FAIL",
        "selection_policy":SEEK_PRESENTATION_POLICY,
        "target_source":"FULL_VIDEO_GENERATED_BOUNDARY_NPZ_PTS",
        "target_quantiles":[0.25,0.50,0.75],
        "records":rec,
        "mae_sec":float(np.mean([z["absolute_drift_sec"] for z in rec])),
        "max_abs_error_sec":float(mx),
        "tolerance_sec":SEEK_ROUNDTRIP_TOL_SEC,
        "fps_arithmetic_used":False,
    }

def runtime_summary(results):
    ok=[r for r in results if r.get('status')!='ERROR' and r.get('elapsed_sec') is not None]
    lat=np.asarray([r['elapsed_sec'] for r in ok],dtype=float); decoded=np.asarray([r.get('frame_count',0) for r in ok],dtype=float)
    return {'videos':len(ok),'latency_p50_sec':float(np.quantile(lat,.50)) if len(lat) else None,'latency_p95_sec':float(np.quantile(lat,.95)) if len(lat) else None,'latency_p99_sec':float(np.quantile(lat,.99)) if len(lat) else None,'throughput_video_per_hour':float(3600/max(1e-9,np.mean(lat))) if len(lat) else None,'decoded_frames_per_sec':float(decoded.sum()/max(1e-9,lat.sum())) if len(lat) else None,'peak_cuda_bytes':int(max((r.get('cuda_peak_bytes',0) for r in ok),default=0))}

In [12]:
# ── CELL FINAL: exact two-GPU execution + full-video PTS evidence ──────────────
videos=[r["video_path"] for r in VIDEO_RECORDS]
if len(videos)!=EXPECTED_VIDEO_COUNT:
    raise RuntimeError(("EXACT_FROZEN_VIDEO_COUNT_REQUIRED",len(videos),EXPECTED_VIDEO_COUNT))
if len(videos)<2:
    raise RuntimeError(("NEED_AT_LEAST_TWO_VIDEOS_FOR_DUAL_GPU_PROOF",len(videos)))

assignments={0:videos[0::2],1:videos[1::2]}

def gpu_worker(gpu_index,paths):
    torch.cuda.set_device(gpu_index)
    device=f"cuda:{gpu_index}"
    detector=AutoShotV2(PRETRAINED_PATH,device)
    out=[]
    for p in paths:
        try:
            out.append(process_video_v2(p,RAW_ROOT,detector,device))
        except Exception as e:
            out.append({
                "video_path":str(p),
                "video_id":Path(p).stem,
                "gpu":device,
                "status":"ERROR",
                "error":repr(e),
            })
    return out

with ThreadPoolExecutor(max_workers=2) as ex:
    f0=ex.submit(gpu_worker,0,assignments[0])
    f1=ex.submit(gpu_worker,1,assignments[1])
    results=f0.result()+f1.result()

used={r.get("gpu") for r in results if r.get("status")!="ERROR"}
if not {"cuda:0","cuda:1"}.issubset(used):
    raise RuntimeError(("DUAL_GPU_PARTICIPATION_NOT_PROVEN",used))

errors=[r for r in results if r.get("status")=="ERROR"]

registry=WORK_ROOT/"shot_interval_registry.jsonl"
with registry.open("w",encoding="utf-8") as f:
    for r in sorted(results,key=lambda x:str(x.get("video_id",""))):
        f.write(json.dumps(r,ensure_ascii=False)+"\n")

# Full-video PTS round-trip sample from generated NPZ, not the first N frames.
sample_probe_video=videos[len(videos)//2]
sample_video_id=Path(sample_probe_video).stem
sample_npz=BOUNDARY_NPZ/f"{sample_video_id}.npz"
if not sample_npz.is_file():
    raise RuntimeError(("SAMPLE_BOUNDARY_NPZ_MISSING",str(sample_npz)))
with np.load(sample_npz) as z:
    sample_full_pts=np.asarray(z["pts_time"],dtype=np.float64)

pts_probe=pts_roundtrip_probe(sample_probe_video,sample_full_pts)
if pts_probe.get("status")=="FAIL":
    raise RuntimeError(("PTS_SEEK_ROUNDTRIP_FAILED",str(sample_probe_video),pts_probe))

runtime=runtime_summary(results)

report={
    "build_version":BUILD_VERSION,
    "stage_id":STAGE_ID,
    "input_lineage":{
        "evidence00_build":E00.get("build_version"),
        "evidence00_freeze_manifest_sha256":sha256_file(E00_FREEZE_PATH),
        "evidence00_snapshot_sha256":sha256_file(SNAPSHOT_PATH),
        "raw_dataset_handle":RAW_DATASET_HANDLE,
        "model_dataset_handle":MODEL_DATASET_HANDLE,
        "checkpoint_path":str(PRETRAINED_PATH),
        "checkpoint_filename":PRETRAINED_PATH.name,
        "checkpoint_source":CHECKPOINT_SOURCE,
        "checkpoint_size_bytes":CHECKPOINT_SIZE_BYTES,
        "checkpoint_sha256":CHECKPOINT_SHA256,
    },
    "videos_expected":EXPECTED_VIDEO_COUNT,
    "videos_processed":len(results)-len(errors),
    "errors":errors,
    "gpu_participation":sorted(used),
    "threshold":SHOT_THRESHOLD,
    "canonical_shot_status":"NOT_CREATED_PROPOSALS_ONLY",
    "accuracy_claim":"NONE_NO_GT",
    "runtime":runtime,
    "sample_pts_seek_roundtrip":pts_probe,
    "validation_state":{
        "STATIC_CONTRACT_VALIDATION":"PASS",
        "CORPUS_RUNTIME_VALIDATION":"PASS" if not errors and len(results)==EXPECTED_VIDEO_COUNT else "FAIL",
        "ACCURACY_VALIDATION":"NOT_RUN",
        "LATENCY_VALIDATION":"MEASURED_CORPUS_RUNTIME_NOT_PROMOTION_GATE",
    },
}

report_path=WORK_ROOT/"run_report.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")

handoff={
    "schema_version":"component-handoff-v1",
    "build_version":BUILD_VERSION,
    "stage_id":STAGE_ID,
    "corpus_id":EXPECTED_CORPUS_ID,
    "dataset_snapshot_id":EXPECTED_DATASET_SNAPSHOT_ID,
    "frame_registry_sha256":EXPECTED_FRAME_REGISTRY_SHA256,
    "parent_evidence00_freeze_sha256":sha256_file(E00_FREEZE_PATH),
    "checkpoint_sha256":CHECKPOINT_SHA256,
    "per_video_root":str(PER_VIDEO),
    "boundary_npz_root":str(BOUNDARY_NPZ),
    "shot_interval_registry":str(registry),
    "run_report":str(report_path),
    "canonical_shot_status":"NOT_CREATED_PROPOSALS_ONLY",
    "next_consumer":"02_BEIT3_ADAPTIVE_KEYFRAME_PTS_v2_2_0",
}
handoff_path=WORK_ROOT/"component01_handoff.json"
handoff_path.write_text(json.dumps(handoff,ensure_ascii=False,indent=2),encoding="utf-8")

print(json.dumps(report,ensure_ascii=False,indent=2))
print("[DONE]",handoff_path)

if errors:
    raise RuntimeError(("AUTOSHOT_PARTIAL_FAILURE",len(errors)))
if len(results)!=EXPECTED_VIDEO_COUNT:
    raise RuntimeError(("AUTOSHOT_VIDEO_COUNT_MISMATCH",len(results),EXPECTED_VIDEO_COUNT))

[PASS] AutoShot checkpoint cuda:0 {'coverage': 1.0, 'keys': 90}
[PASS] AutoShot checkpoint cuda:1 {'coverage': 1.0, 'keys': 90}


/tmp/ipykernel_24/169253098.py:189: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  output = params[indices]  # (num_samples, ...)


{
  "build_version": "aic2026-autoshot-pts-streaming-v2.2.3",
  "stage_id": "component01_autoshot_pts_streaming",
  "input_lineage": {
    "evidence00_build": "aic2026-evidence-frozen-system-snapshot-v0.2.1",
    "evidence00_freeze_manifest_sha256": "c5aa00078be9b45c9a4d8c7161fb46d69245215be6466c30497307ca62be37c2",
    "evidence00_snapshot_sha256": "d18c6e5731bf610e55546efe7ffb68565a8e19000633baee1608a516933048e0",
    "raw_dataset_handle": "nguyenminhtric/data-batch-1-aic",
    "model_dataset_handle": "anhnguynnhtinh/model-weights-autoshot",
    "checkpoint_path": "/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot/ckpt_0_200_0.pth",
    "checkpoint_filename": "ckpt_0_200_0.pth",
    "checkpoint_source": "OFFICIAL_BASELINE_FILENAME",
    "checkpoint_size_bytes": 57243097,
    "checkpoint_sha256": "3e85290546ce6d32f4a3581ec2cae87aedd2402246a0d46b4d361a330b4b1fa6"
  },
  "videos_expected": 873,
  "videos_processed": 873,
  "errors": [],
  "gpu_participation": [
    "cuda:0",
